# Patient bootstrap and SAE feature stability

No model training. Analyze the 36 completed native baseline/AuxK SAE models.
Inputs: the SAME `rank_control_embeddings_v1.zip`, plus
`auxk_native_controls_v1_enc10-42-52_ALL_RESULTS.zip` (79.7 MB).
Upload this completed-model ZIP ONCE to the root of your private
`YOUR_USERNAME/YOUR_PRIVATE_DATASET` repository, keeping its filename. Then use the same
read-only HF_TOKEN. The notebook verifies both source archive hashes.

Default: all encoder seeds 10/42/52, both methods, baseline and AuxK. Output:
`patient_stability_v1`. No new encoder/saved-model analyses and no raw MRI PKL needed.

Patient bootstrap: 1,000 whole-patient resamples, paired across all methods/models.
Resample 37 validation patients with replacement, retaining all their slices.
Recompute feature eligibility and top-10 label purity in every resample; do not
bootstrap slices independently. Patient-distinct top-10 treats repeated copies of
one original patient as one identity. Recompute eligible coverage in both metrics.
Shared patient draws give paired NCL-minus-SimCLR and AuxK-minus-baseline intervals.
Average SAE seeds within encoder, then average the three fixed encoder seeds.
Intervals are descriptive 95% percentile intervals conditional on these trained
models; they do not include encoder-training uncertainty or undo validation reuse.
No bootstrap-adjusted-purity intervals or spatial Dice are calculated here.

Feature stability: within each encoder/method/variant, compare all three pairs of
SAE seeds. Select at most 4,096 training slices by patient-balanced round robin,
without labels. Fit one-to-one feature matches to training activation correlations
only, among features with >=10 positive training-subset examples and nonzero
variance. Assess matched activation correlations and support Jaccard on validation;
also report decoder cosine, matching coverage and a random-matching reference.
Thresholds 0.7/0.9 are descriptive sensitivity cutoffs, not validated quality levels.
Unmatched, constant and ineligible features remain visible in coverage counts.
This tests within-encoder dictionary reproducibility, not cross-encoder neuron
identity, causal concepts or clinical relevance.

Use Apply settings -> Download / verify embedding bundle -> Start / continue
controls. Live progress refreshes every 10 seconds. Create full results ZIP after
the worker stops. Export contains analysis tables/replicates, not the input models.
Keep your original model archive. Interrupted analyses reuse committed outputs.
All results use previously inspected validation data; no held-out test is read.


Local notebook: use the ncl_sae kernel. Run All starts the full registered plan. Edit SELECTED_SEEDS before running if splitting sessions.

In [ ]:
from pathlib import Path
import types,json,hashlib
RUNTIME_SOURCE='"""Expanded-cohort rank controls using archived CLS embeddings only."""\nfrom pathlib import Path\nimport gc,hashlib,itertools,json,os,random,sys,time,zipfile\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nDEFAULT_PLAN={\'version\':\'expanded-rank-control-v1\',\'methods\':[\'ncl\',\'simclr\'],\n \'encoder_seeds\':[10,42,52],\'sae_seeds\':[10,42,52],\'ranks\':[8],\n \'conditions\':[\'native\',\'pca_rank\',\'spectrum_matched\'],\'dictionary_size\':1536,\'k\':24,\n \'epochs\':30,\'batch_size\':512,\'lr\':0.0005,\'gradient_clip\':1.0,\'top_m\':10,\n \'permutations\':100,\'probe_C\':1.0,\'probe_max_iter\':2000,\n \'spectrum_reference\':\'NCL training covariance, same encoder seed\',\n \'save_completed_models\':True,\'per_epoch_checkpoints\':False,\n \'scope\':\'exploratory validation follow-up; no test images/features or spatial maps\'}\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef replace_file(source,target):\n    # Windows readers may briefly deny replacement while the live UI reads CSVs.\n    for attempt in range(41):\n        try:\n            os.replace(source,target);return\n        except PermissionError:\n            if os.name!=\'nt\' or attempt==40:raise\n            time.sleep(.05)\n\ndef environment():\n    import sklearn\n    return {\'python\':sys.version,\'torch\':str(torch.__version__),\'numpy\':np.__version__,\n        \'pandas\':pd.__version__,\'scikit_learn\':sklearn.__version__,\'cuda\':torch.version.cuda,\n        \'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}\n\ndef atomic_json(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.\'+str(os.getpid())+\'.tmp\')\n    tmp.write_text(json.dumps(value,indent=2,allow_nan=False),encoding=\'utf-8\');replace_file(tmp,path)\n\ndef atomic_csv(frame,path):\n    path=Path(path);tmp=path.with_name(path.name+\'.\'+str(os.getpid())+\'.tmp\')\n    frame.to_csv(tmp,index=False);replace_file(tmp,path)\n\ndef bundle_download(repo_id,filename,destination,expected_sha):\n    target=Path(destination).expanduser().resolve();target.parent.mkdir(parents=True,exist_ok=True)\n    if target.is_file():\n        if sha(target)!=expected_sha:raise ValueError(\'Input bundle SHA256 mismatch\')\n        return str(target)\n    token=os.environ.get(\'HF_TOKEN\',\'\').strip()\n    if not token:\n        from dotenv import dotenv_values\n        token=str(dotenv_values(\'.env\').get(\'HF_TOKEN\') or \'\').strip()\n    if not token:raise RuntimeError(\'Set read-only HF_TOKEN in runtime secrets or .env\')\n    from huggingface_hub import hf_hub_download\n    downloaded=Path(hf_hub_download(repo_id=repo_id,repo_type=\'dataset\',filename=filename,token=token,local_dir=str(target.parent))).resolve()\n    if sha(downloaded)!=expected_sha:raise ValueError(\'Input bundle SHA256 mismatch\')\n    if downloaded!=target:os.replace(downloaded,target)\n    return str(target)\n\nclass Inputs:\n    def __init__(self,bundle,cache,expected_sha):\n        self.bundle=Path(bundle).resolve();self.cache=Path(cache).resolve();self.cache.mkdir(parents=True,exist_ok=True)\n        signature={\'bytes\':self.bundle.stat().st_size,\'mtime_ns\':self.bundle.stat().st_mtime_ns,\'sha256\':expected_sha}\n        receipt=self.cache/\'bundle_verified.json\'\n        if not receipt.exists() or json.loads(receipt.read_text())!=signature:\n            if sha(self.bundle)!=expected_sha:raise ValueError(\'Wrong/corrupt embedding bundle\')\n            with zipfile.ZipFile(self.bundle) as z:\n                self.registration=json.loads(z.read(\'input_manifest.json\'))\n                if self.registration[\'format\']!=\'rank-control-embeddings-v1\':raise ValueError(\'Wrong bundle format\')\n                for name,identity in self.registration[\'files\'].items():\n                    target=(self.cache/name).resolve()\n                    if not target.is_relative_to(self.cache):raise ValueError(\'Unsafe archive path\')\n                    target.parent.mkdir(parents=True,exist_ok=True)\n                    temp=target.with_name(target.name+\'.tmp\');h=hashlib.sha256();count=0\n                    with z.open(name) as src,open(temp,\'wb\') as dst:\n                        while True:\n                            b=src.read(8*1024*1024)\n                            if not b:break\n                            h.update(b);count+=len(b);dst.write(b)\n                    if count!=identity[\'bytes\'] or h.hexdigest()!=identity[\'sha256\']:raise ValueError(\'Corrupt embedding file: \'+name)\n                    os.replace(temp,target)\n                atomic_json(self.cache/\'input_manifest.json\',self.registration)\n            atomic_json(receipt,signature)\n        self.registration=json.loads((self.cache/\'input_manifest.json\').read_text())\n        if sha(self.cache/\'split_manifest.csv\')!=self.registration[\'manifest_sha256\']:raise ValueError(\'Manifest changed\')\n        self.frame=pd.read_csv(self.cache/\'split_manifest.csv\',dtype={\'patient_id\':str,\'sample_id\':str})\n        if self.frame.duplicated([\'patient_id\',\'slice_idx\']).any() or self.frame.groupby(\'patient_id\').split.nunique().max()!=1:\n            raise ValueError(\'Duplicate samples or crossing patient partitions\')\n        self.pca_cache={}\n\n    def load(self,mode,seed,split):\n        if split not in (\'train\',\'val\'):raise ValueError(\'No test features available\')\n        prefix=self.cache/\'runs\'/mode/f\'encoder_{seed}\'\n        relative=f\'runs/{mode}/encoder_{seed}/{split}.npz\'\n        if sha(prefix/f\'{split}.npz\')!=self.registration[\'files\'][relative][\'sha256\']:raise ValueError(\'Cached input file changed\')\n        meta=json.loads((prefix/f\'{split}_identity.json\').read_text())\n        expected=self.frame.loc[self.frame.split==split,\'sample_id\'].astype(str).to_numpy()\n        if meta[\'manifest\']!=self.registration[\'manifest_sha256\'] or meta[\'samples\']!=hashlib.sha256(\'\\n\'.join(expected).encode()).hexdigest():\n            raise ValueError(\'Embedding identity mismatch\')\n        with np.load(prefix/f\'{split}.npz\',allow_pickle=False) as data:\n            x=data[\'x\'].astype(np.float32);ids=data[\'sample_ids\']\n        if not np.array_equal(ids,expected) or x.shape!=(len(expected),192) or not np.isfinite(x).all():raise ValueError(\'Invalid embedding cache\')\n        return x\n\n    def pca(self,mode,seed,x=None):\n        key=(mode,seed)\n        if key not in self.pca_cache:\n            x=self.load(mode,seed,\'train\') if x is None else x\n            mean=x.astype(np.float64).mean(0);cov=covariance(x)\n            eig,basis=np.linalg.eigh(cov);order=np.argsort(eig)[::-1]\n            self.pca_cache[key]={\'mean\':mean.astype(np.float32),\'eigenvalues\':np.maximum(eig[order],0),\'basis\':basis[:,order]}\n        return self.pca_cache[key]\n\ndef covariance(x):\n    mean=np.mean(x,axis=0,dtype=np.float64);cov=np.zeros((x.shape[1],x.shape[1]),dtype=np.float64)\n    for start in range(0,len(x),8192):\n        centered=x[start:start+8192].astype(np.float64)-mean;cov+=centered.T@centered\n    return cov/max(1,len(x)-1)\n\ndef spectrum_stats(eigenvalues):\n    v=np.maximum(eigenvalues,0);total=float(v.sum());p=v/total if total else v\n    p=p[p>0]\n    return {\'variance\':total,\'effective_rank\':float(np.exp(-np.sum(p*np.log(p)))) if total else 0,\n            \'participation_ratio\':float(total**2/np.sum(v*v)) if total else 0}\n\ndef build_transform(inputs,mode,seed,condition,rank,train,val):\n    info=inputs.pca(mode,seed,train);mean=info[\'mean\'];centered_train=train-mean;centered_val=val-mean\n    if condition==\'native\':\n        basis=np.eye(192);scale=np.ones(192);rank=192\n        ztrain=centered_train;zval=centered_val;target=None;pca_ev=1.0\n    else:\n        basis=info[\'basis\'][:,:rank];values=info[\'eigenvalues\'][:rank]\n        if np.min(values)<=info[\'eigenvalues\'][0]*1e-12:raise ValueError(\'Requested rank contains numerically singular dimensions\')\n        scale=np.ones(rank)\n        target=None\n        if condition==\'spectrum_matched\':\n            target=inputs.pca(\'ncl\',seed)[\'eigenvalues\'][:rank]\n            if np.min(target)<=0:raise ValueError(\'NCL reference covariance is singular\')\n            scale=np.sqrt(target/values)\n        elif condition!=\'pca_rank\':raise ValueError(\'Unknown control condition\')\n        ztrain=((centered_train@basis)*scale)@basis.T;zval=((centered_val@basis)*scale)@basis.T\n        unscaled=(centered_val@basis)@basis.T\n        pca_ev=1-float(np.sum((centered_val.astype(np.float64)-unscaled)**2))/float(np.sum(centered_val.astype(np.float64)**2))\n    ztrain=np.asarray(ztrain,dtype=np.float32);zval=np.asarray(zval,dtype=np.float32)\n    train_eig=np.maximum(np.linalg.eigvalsh(covariance(ztrain)),0)[::-1]\n    val_eig=np.maximum(np.linalg.eigvalsh(covariance(zval)),0)[::-1]\n    diagnostics={\'condition\':condition,\'rank\':rank,\'PCA_only_validation_EV\':pca_ev,\n        \'PCA_train_variance_retained\':float(info[\'eigenvalues\'][:rank].sum()/info[\'eigenvalues\'].sum()),\n        \'train\':spectrum_stats(train_eig),\'validation\':spectrum_stats(val_eig),\n        \'matching_scope\':\'train only; validation spectra are measured, never forced\'}\n    if target is not None:\n        error=float(np.linalg.norm(train_eig[:rank]-target)/np.linalg.norm(target))\n        if error>1e-4:raise ValueError(\'Training spectrum matching failed\')\n        diagnostics.update(target=spectrum_stats(target),relative_training_spectrum_error=error)\n    recipe={\'mean\':mean,\'basis\':basis.astype(np.float32),\'scale\':scale.astype(np.float32),\'condition\':condition,\'rank\':rank}\n    return ztrain,zval,recipe,diagnostics\n\nclass SAE(nn.Module):\n    def __init__(self,width,dictionary,k):\n        super().__init__();self.k=k;self.pre_bias=nn.Parameter(torch.zeros(width))\n        self.encoder=nn.Linear(width,dictionary);self.decoder=nn.Linear(dictionary,width,bias=False)\n    def forward(self,x):\n        raw=F.relu(self.encoder(x-self.pre_bias));v,i=raw.topk(self.k,dim=-1)\n        acts=torch.zeros_like(raw).scatter_(-1,i,v)\n        return self.decoder(acts)+self.pre_bias,acts\n\ndef seed_all(seed):\n    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False\n\ndef purity_scores(top,labels,classes):\n    if len(top)==0:return np.empty(0),np.empty(0)\n    counts=np.stack([(labels[top]==label).sum(axis=1) for label in classes],axis=1)\n    p=counts/top.shape[1];entropy=-np.sum(np.where(p>0,p*np.log(np.maximum(p,1e-300)),0),axis=1)\n    return p.max(axis=1),entropy\n\ndef selectivity(acts,labels,patients,m,permutations,seed):\n    support=(acts>0).sum(0);alive=np.flatnonzero(support);eligible=np.flatnonzero(support>=m)\n    top=np.argsort(acts[:,eligible],axis=0)[-m:].T if len(eligible) else np.empty((0,m),int)\n    classes=np.unique(labels);purity,entropy=purity_scores(top,labels,classes)\n    patient_top=[];patient_features=[]\n    for feature in eligible:\n        chosen=[];seen=set()\n        for index in np.argsort(acts[:,feature])[::-1]:\n            if acts[index,feature]<=0:break\n            if patients[index] in seen:continue\n            seen.add(patients[index]);chosen.append(index)\n            if len(chosen)==m:break\n        if len(chosen)==m:patient_features.append(feature);patient_top.append(chosen)\n    patient_top=np.asarray(patient_top,dtype=int).reshape(-1,m)\n    pp,pe=purity_scores(patient_top,labels,classes)\n    rng=np.random.default_rng(seed);null=[];pnull=[]\n    for _ in range(permutations):\n        shuffled=rng.permutation(labels)\n        a,_=purity_scores(top,shuffled,classes);b,_=purity_scores(patient_top,shuffled,classes)\n        if len(a):null.append(float(a.mean()))\n        if len(b):pnull.append(float(b.mean()))\n    def average(v):return float(np.mean(v)) if len(v) else None\n    result={\'alive_features\':len(alive),\'eligible_features\':len(eligible),\'mean_purity\':average(purity),\n        \'mean_entropy_nats\':average(entropy),\'permutation_mean_purity\':average(null),\n        \'patient_distinct_eligible_features\':len(patient_features),\'patient_distinct_purity\':average(pp),\n        \'patient_distinct_permutation_mean_purity\':average(pnull),\'patient_distinct_entropy\':average(pe),\n        \'mean_positive_features_per_slice\':float((acts>0).sum(1).mean()),\n        \'permutation_scope\':\'descriptive slice-label shuffle; not a patient-level significance test\'}\n    result[\'purity_minus_permutation\']=result[\'mean_purity\']-result[\'permutation_mean_purity\'] if null else None\n    result[\'patient_distinct_purity_minus_permutation\']=result[\'patient_distinct_purity\']-result[\'patient_distinct_permutation_mean_purity\'] if pnull else None\n    table=pd.DataFrame({\'feature\':eligible,\'positive_support\':support[eligible],\'purity\':purity,\'entropy_nats\':entropy})\n    patient_table=pd.DataFrame({\'feature\':patient_features,\'purity\':pp,\'entropy_nats\':pe})\n    return result,table,patient_table\n\ndef evaluate(model,zval,original_val,recipe,labels,patients,plan,sae_seed):\n    device=next(model.parameters()).device;codes=[];sse=0.;original_sse=0.\n    mean=recipe[\'mean\'];basis=recipe[\'basis\'];scale=recipe[\'scale\']\n    centered=original_val.astype(np.float64)-mean\n    with torch.no_grad():\n        for start in range(0,len(zval),512):\n            x=zval[start:start+512];recon,acts=model(torch.from_numpy(x).to(device))\n            r=recon.cpu().numpy();codes.append(acts.cpu().numpy())\n            sse+=float(np.sum((x.astype(np.float64)-r)**2))\n            restored=r if recipe[\'condition\']==\'native\' else ((r@basis)/scale)@basis.T\n            original_sse+=float(np.sum((centered[start:start+len(x)]-restored)**2))\n    acts=np.concatenate(codes);reference=float(np.sum(zval.astype(np.float64)**2))\n    metrics={\'normalized_SAE_EV\':1-sse/reference,\'original_coordinate_EV\':1-original_sse/float(np.sum(centered**2)),\'mse\':sse/zval.size}\n    concept,features,patient_features=selectivity(acts,labels,patients,plan[\'top_m\'],plan[\'permutations\'],sae_seed)\n    metrics.update(concept)\n    return metrics,features,patient_features\n\ndef fit_one(ztrain,zval,original_val,recipe,diagnostics,labels,patients,plan,mode,encoder_seed,sae_seed,folder,identity):\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    protocol={**identity,\'method\':mode,\'encoder_seed\':encoder_seed,\'sae_seed\':sae_seed,\n        \'condition\':recipe[\'condition\'],\'rank\':recipe[\'rank\'],\'plan\':plan}\n    registration=folder/\'protocol.json\'\n    if registration.exists() and json.loads(registration.read_text())!=protocol:raise ValueError(\'Fit protocol changed; choose new output\')\n    atomic_json(registration,protocol)\n    complete=folder/\'validation_metrics.json\';model_path=folder/\'completed_model.pt\'\n    if complete.exists():\n        r=json.loads(complete.read_text())\n        if not model_path.exists() or sha(model_path)!=r[\'model_sha256\']:raise ValueError(\'Missing/corrupt completed dictionary\')\n        print(\'Reusing completed control:\',mode,encoder_seed,sae_seed,recipe[\'condition\'],recipe[\'rank\'],flush=True)\n        return r\n    seed_all(sae_seed);device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model=SAE(ztrain.shape[1],plan[\'dictionary_size\'],plan[\'k\']).to(device)\n    if model_path.exists():\n        saved=torch.load(model_path,map_location=\'cpu\',weights_only=True)\n        if saved[\'protocol\']!=protocol:raise ValueError(\'Completed model protocol mismatch\')\n        model.load_state_dict(saved[\'model\']);history=saved[\'history\']\n    else:\n        optimizer=torch.optim.Adam(model.parameters(),lr=plan[\'lr\']);history=[]\n        loader=torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.from_numpy(ztrain)),batch_size=plan[\'batch_size\'],shuffle=True)\n        for epoch in range(plan[\'epochs\']):\n            model.train();total=count=0\n            for (batch,) in loader:\n                batch=batch.to(device);optimizer.zero_grad(set_to_none=True);recon,_=model(batch)\n                loss=F.mse_loss(recon,batch)\n                if not torch.isfinite(loss):raise ValueError(\'Non-finite loss\')\n                loss.backward();nn.utils.clip_grad_norm_(model.parameters(),plan[\'gradient_clip\']);optimizer.step()\n                total+=float(loss.detach())*len(batch);count+=len(batch)\n            history.append({\'epoch\':epoch+1,\'train_mse\':total/count})\n            atomic_json(folder/\'history.json\',history)\n            print(mode,encoder_seed,recipe[\'condition\'],recipe[\'rank\'],\'SAE\',sae_seed,\'epoch\',epoch+1,\'mse\',total/count,flush=True)\n        # One final artifact, no optimizer/scaler/RNG or per-epoch model writes.\n        payload={\'model\':{k:v.detach().cpu() for k,v in model.state_dict().items()},\'protocol\':protocol,\'history\':history,\'training_environment\':environment(),\n            \'transform\':{k:torch.from_numpy(v) if isinstance(v,np.ndarray) else v for k,v in recipe.items()}}\n        temp=model_path.with_suffix(\'.tmp\');torch.save(payload,temp);replace_file(temp,model_path)\n    model.eval();metrics,features,patient_features=evaluate(model,zval,original_val,recipe,labels,patients,plan,sae_seed)\n    features.to_csv(folder/\'validation_features.csv\',index=False);patient_features.to_csv(folder/\'patient_distinct_features.csv\',index=False)\n    result={**metrics,\'method\':mode,\'encoder_seed\':encoder_seed,\'sae_seed\':sae_seed,\'condition\':recipe[\'condition\'],\n        \'rank\':recipe[\'rank\'],\'k\':plan[\'k\'],\'epochs\':plan[\'epochs\'],\'split\':\'val\',\'diagnostics\':diagnostics,\n        \'model_sha256\':sha(model_path),\'selection_scope\':\'exploratory controls; no test evaluation\'}\n    atomic_json(complete,result);del model;gc.collect()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n    return result\n\ndef probe(ztrain,zval,labels_train,labels_val,plan):\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler\n    from sklearn.linear_model import LogisticRegression\n    model=make_pipeline(StandardScaler(),LogisticRegression(C=plan[\'probe_C\'],max_iter=plan[\'probe_max_iter\']))\n    model.fit(ztrain,labels_train);predictions=model.predict(zval)\n    return {\'slice_accuracy\':float(np.mean(predictions==labels_val)),\'train_fit_only\':True,\'C\':plan[\'probe_C\'],\'max_iter\':plan[\'probe_max_iter\']}\n\ndef run(bundle,bundle_sha,output,plan=None,seeds=None,methods=None,max_hours=8,require_cuda=True):\n    plan=json.loads(json.dumps(DEFAULT_PLAN if plan is None else plan));seeds=plan[\'encoder_seeds\'] if seeds is None else list(seeds)\n    methods=plan[\'methods\'] if methods is None else list(methods)\n    if not seeds or set(seeds)-set(plan[\'encoder_seeds\']) or not methods or set(methods)-set(plan[\'methods\']):raise ValueError(\'Invalid method/seed selection\')\n    if require_cuda and not torch.cuda.is_available():raise RuntimeError(\'Attach a CUDA GPU\')\n    root=Path(output).expanduser().resolve();root.mkdir(parents=True,exist_ok=True)\n    started=time.monotonic();inputs=Inputs(bundle,root/\'input_cache\',bundle_sha)\n    identity={\'bundle_sha256\':bundle_sha,\'manifest_sha256\':inputs.registration[\'manifest_sha256\'],\n        \'runtime_sha256\':globals().get(\'RUNTIME_SHA256\',\'development\'),\'plan\':plan}\n    registration=root/\'protocol.json\'\n    if registration.exists() and json.loads(registration.read_text())!=identity:raise ValueError(\'Control protocol/code changed; choose a new output folder\')\n    atomic_json(registration,identity)\n    atomic_json(root/\'environment.json\',environment())\n    val=inputs.frame[inputs.frame.split==\'val\'];train_frame=inputs.frame[inputs.frame.split==\'train\']\n    labels=val.dominant_label.to_numpy();patients=val.patient_id.to_numpy()\n    status=\'completed selected controls\'\n    for seed,mode in itertools.product(seeds,methods):\n        train=inputs.load(mode,seed,\'train\');validation=inputs.load(mode,seed,\'val\')\n        cases=[(\'native\',192)]+[(condition,rank) for condition in plan[\'conditions\'] if condition!=\'native\' for rank in plan[\'ranks\']]\n        for condition,rank in cases:\n            if (root/\'stop_after_fit.flag\').exists() or time.monotonic()-started>=max_hours*3600:\n                status=\'stopped between fits; completed dictionaries retained\';break\n            ztrain,zval,recipe,diagnostics=build_transform(inputs,mode,seed,condition,rank,train,validation)\n            case_dir=root/\'fits\'/mode/f\'encoder_{seed}\'/f\'{condition}_r{rank}\'\n            case_dir.mkdir(parents=True,exist_ok=True);atomic_json(case_dir/\'transform_diagnostics.json\',diagnostics)\n            probe_path=case_dir/\'validation_probe.json\'\n            if not probe_path.exists():atomic_json(probe_path,probe(ztrain,zval,train_frame.has_tumor.to_numpy(),val.has_tumor.to_numpy(),plan))\n            for sae_seed in plan[\'sae_seeds\']:\n                if (root/\'stop_after_fit.flag\').exists() or time.monotonic()-started>=max_hours*3600:\n                    status=\'stopped between fits; completed dictionaries retained\';break\n                fit_one(ztrain,zval,validation,recipe,diagnostics,labels,patients,plan,mode,seed,sae_seed,case_dir/f\'sae_{sae_seed}\',identity)\n                summarize(root)\n            del ztrain,zval;gc.collect()\n            if status!=\'completed selected controls\':break\n        del train,validation;gc.collect()\n        if status!=\'completed selected controls\':break\n    report={\'status\':status,\'encoder_seeds\':seeds,\'methods\':methods,\'elapsed_hours\':(time.monotonic()-started)/3600,\n        \'test_evaluated\':False,\'encoders_retrained\':False,\'per_epoch_checkpoints\':False,\'completed_models_saved_once\':True}\n    atomic_json(root/\'last_session.json\',report);summarize(root)\n    return report\n\ndef summarize(root):\n    root=Path(root);records=[json.loads(p.read_text()) for p in root.glob(\'fits/*/encoder_*/*/sae_*/validation_metrics.json\')]\n    if not records:return pd.DataFrame(),pd.DataFrame(),pd.DataFrame()\n    frame=pd.json_normalize(records);group=[\'method\',\'encoder_seed\',\'condition\',\'rank\']\n    metrics=[\'normalized_SAE_EV\',\'original_coordinate_EV\',\'alive_features\',\'eligible_features\',\'mean_purity\',\'purity_minus_permutation\',\n        \'patient_distinct_purity\',\'patient_distinct_purity_minus_permutation\',\'patient_distinct_eligible_features\']\n    means=frame.groupby(group)[metrics].mean().reset_index()\n    means[\'sae_replicates\']=frame.groupby(group).size().to_numpy()\n    # Only complete sets of SAE seeds enter the across-encoder summary.\n    plan=json.loads((root/\'protocol.json\').read_text())[\'plan\'];complete=means[means.sae_replicates==len(plan[\'sae_seeds\'])]\n    summary=complete.groupby([\'method\',\'condition\',\'rank\'])[metrics].agg([\'mean\',\'std\',\'count\']).reset_index()\n    summary.columns=[\'_\'.join(str(v) for v in c if str(v)) if isinstance(c,tuple) else c for c in summary.columns]\n    atomic_csv(frame,root/\'all_control_fits.csv\');atomic_csv(means,root/\'per_encoder_controls.csv\');atomic_csv(summary,root/\'control_summary.csv\')\n    differences=[]\n    for (seed,condition,rank),g in complete.groupby([\'encoder_seed\',\'condition\',\'rank\']):\n        if set(g.method)!={\'ncl\',\'simclr\'}:continue\n        a=g[g.method==\'ncl\'].iloc[0];b=g[g.method==\'simclr\'].iloc[0]\n        differences.append({\'encoder_seed\':int(seed),\'condition\':condition,\'rank\':int(rank),\n            **{\'ncl_minus_simclr_\'+m:float(a[m]-b[m]) for m in metrics}})\n    atomic_csv(pd.DataFrame(differences),root/\'paired_control_differences.csv\')\n    return frame,means,summary\n\ndef export_results(root):\n    root=Path(root).resolve()\n    seeds=sorted({json.loads(p.read_text())[\'encoder_seed\'] for p in root.glob(\'fits/*/encoder_*/*/sae_*/validation_metrics.json\')})\n    suffix=\'_enc\'+\'-\'.join(str(s) for s in seeds) if seeds else \'\'\n    target=root.parent/(root.name+suffix+\'_ALL_RESULTS.zip\')\n    with zipfile.ZipFile(target,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(root.rglob(\'*\')):\n            if not p.is_file() or \'input_cache\' in p.relative_to(root).parts or p.name in (\'.env\',\'worker_request.json\'):continue\n            if p.suffix in (\'.json\',\'.csv\',\'.pt\',\'.log\',\'.txt\',\'.png\'):z.write(p,p.relative_to(root))\n    return str(target)\n\n"""Append to rank runtime: paired patient-cluster bootstrap and train-only matching."""\nimport io\nfrom scipy.optimize import linear_sum_assignment\nDEFAULT_PLAN={\'version\':\'patient-stability-v1\',\'encoder_seeds\':[10,42,52],\n \'methods\':[\'ncl\',\'simclr\'],\'sae_seeds\':[10,42,52],\'conditions\':[\'baseline\',\'auxk\'],\n \'bootstrap_replicates\':1000,\'bootstrap_seed\':271027,\'bootstrap_batch\':8,\'top_m\':10,\n \'alignment_train_slices\':4096,\'alignment_seed\':271028,\'alignment_min_positive_support\':10,\n \'correlation_thresholds\':[0.7,0.9],\'random_alignment_replicates\':100,\n \'models_filename\':\'auxk_native_controls_v1_enc10-42-52_ALL_RESULTS.zip\',\n \'models_sha256\':\'724f0a4e5d81b1bcebca019bdf17bb1eeb673ce7d1d8640960909e82fd165856\',\n \'scope\':\'exploratory validation; percentile patient-bootstrap conditional on fixed trained models; no test\'}\nBOOT_METRICS=[\'mean_purity\',\'patient_distinct_purity\',\'normalized_SAE_EV\',\'alive_features\',\'eligible_features\',\'patient_distinct_eligible_features\']\n\ndef balanced_train_indices(frame,limit,seed):\n    rng=np.random.default_rng(seed)\n    groups=[rng.permutation(np.flatnonzero(frame.patient_id.to_numpy()==p)).tolist() for p in sorted(frame.patient_id.unique())]\n    rng.shuffle(groups);selected=[]\n    while len(selected)<min(limit,len(frame)):\n        for group in groups:\n            if group:selected.append(group.pop())\n            if len(selected)==min(limit,len(frame)):break\n    return np.asarray(selected,dtype=int)\n\ndef encode(model,x):\n    device=next(model.parameters()).device;codes=[];errors=[]\n    with torch.no_grad():\n        for start in range(0,len(x),512):\n            batch=torch.from_numpy(x[start:start+512]).to(device)\n            recon,acts=model(batch)\n            codes.append(acts.cpu().numpy())\n            errors.append(((batch-recon).double().square().sum(1)).cpu().numpy())\n    return np.concatenate(codes),np.concatenate(errors)\n\ndef patient_pool(codes,labels,patients,errors,reference,m,device):\n    unique=sorted(set(patients));width=codes.shape[1]\n    values=np.full((len(unique),m,width),-np.inf,dtype=np.float32)\n    lab=np.zeros((len(unique),m,width),dtype=np.int64)\n    support=[];sse=[];ref=[]\n    # Per-patient top M is sufficient for every bootstrap global top M.\n    for p,patient in enumerate(unique):\n        indices=np.flatnonzero(patients==patient);a=codes[indices]\n        order=np.argsort(a,axis=0,kind=\'stable\')[-m:][::-1]\n        v=np.take_along_axis(a,order,axis=0)\n        count=len(v);values[p,:count]=np.where(v>0,v,-np.inf)\n        lab[p,:count]=labels[indices[order]]\n        support.append((a>0).sum(0));sse.append(errors[indices].sum());ref.append(reference[indices].sum())\n    def tensor(x,dtype=None):return torch.as_tensor(x,device=device,dtype=dtype)\n    return {\'values\':tensor(values),\'labels\':tensor(lab),\'support\':tensor(np.asarray(support),torch.float64),\n        \'sse\':tensor(sse,torch.float64),\'reference\':tensor(ref,torch.float64),\n        \'classes\':np.unique(labels).tolist(),\'patients\':unique}\n\ndef bootstrap_statistics(pool,draws,m,batch_size=8):\n    values=pool[\'values\'];labels=pool[\'labels\'];device=values.device;P=values.shape[0]\n    rows=[]\n    for start in range(0,len(draws),batch_size):\n        index=torch.as_tensor(draws[start:start+batch_size],device=device,dtype=torch.long);B=len(index)\n        weights=torch.zeros((B,P),device=device,dtype=torch.float64)\n        weights.scatter_add_(1,index,torch.ones_like(index,dtype=torch.float64))\n        support=weights@pool[\'support\'];eligible=support>=m\n        candidates=values[index].reshape(B,-1,values.shape[-1])\n        chosen=candidates.topk(m,dim=1).indices\n        selected_labels=labels[index].reshape(B,-1,values.shape[-1]).gather(1,chosen)\n        counts=torch.stack([(selected_labels==c).sum(1) for c in pool[\'classes\']],dim=1)\n        purity=counts.amax(1).double()/m\n        raw=(purity*eligible).sum(1)/eligible.sum(1).clamp_min(1)\n        raw=raw.masked_fill(eligible.sum(1)==0,float(\'nan\'))\n        # Repeated copies of one patient do not count as different identities.\n        present=weights>0\n        best=values[:,0,:][None,:,:].expand(B,-1,-1).masked_fill(~present[:,:,None],-np.inf)\n        distinct_support=torch.isfinite(best).sum(1);distinct_eligible=distinct_support>=m\n        picked=best.topk(m,dim=1).indices\n        pl=labels[:,0,:][None,:,:].expand(B,-1,-1).gather(1,picked)\n        pc=torch.stack([(pl==c).sum(1) for c in pool[\'classes\']],dim=1)\n        dp=(pc.amax(1).double()/m*distinct_eligible).sum(1)/distinct_eligible.sum(1).clamp_min(1)\n        dp=dp.masked_fill(distinct_eligible.sum(1)==0,float(\'nan\'))\n        ev=1-(weights@pool[\'sse\'])/(weights@pool[\'reference\'])\n        rows.append(torch.stack([raw,dp,ev,(support>0).sum(1),eligible.sum(1),distinct_eligible.sum(1)],dim=1).cpu().numpy())\n    return np.concatenate(rows)\n\ndef normalized_columns(a):\n    a=a.astype(np.float64);a=a-a.mean(0);norm=np.sqrt(np.sum(a*a,axis=0))\n    return (a/np.maximum(norm,1e-30)).astype(np.float32),norm>1e-12\n\ndef feature_match(train_a,train_b,val_a,val_b,decoder_a,decoder_b,plan,device):\n    na,oka=normalized_columns(train_a);nb,okb=normalized_columns(train_b)\n    ia=np.flatnonzero(oka&((train_a>0).sum(0)>=plan[\'alignment_min_positive_support\']))\n    ib=np.flatnonzero(okb&((train_b>0).sum(0)>=plan[\'alignment_min_positive_support\']))\n    if not len(ia) or not len(ib):return pd.DataFrame(),{\'eligible_left\':len(ia),\'eligible_right\':len(ib),\'matches\':0}\n    score=(torch.from_numpy(na[:,ia]).to(device).T@torch.from_numpy(nb[:,ib]).to(device)).cpu().numpy()\n    left,right=linear_sum_assignment(score,maximize=True);a=ia[left];b=ib[right]\n    va,valid_a=normalized_columns(val_a);vb,valid_b=normalized_columns(val_b)\n    valid=valid_a[a]&valid_b[b]\n    corr=np.sum(va[:,a]*vb[:,b],axis=0).astype(float);corr[~valid]=np.nan\n    da=decoder_a[:,a];db=decoder_b[:,b]\n    cosine=np.sum(da*db,axis=0)/np.maximum(np.linalg.norm(da,axis=0)*np.linalg.norm(db,axis=0),1e-30)\n    pa=val_a[:,a]>0;pb=val_b[:,b]>0;union=(pa|pb).sum(0)\n    jaccard=np.divide((pa&pb).sum(0),union,out=np.full(len(a),np.nan),where=union>0)\n    table=pd.DataFrame({\'left_feature\':a,\'right_feature\':b,\'train_correlation\':score[left,right],\n        \'validation_correlation\':corr,\'decoder_cosine\':cosine,\'validation_activation_jaccard\':jaccard,\n        \'validation_positive_left\':pa.sum(0),\'validation_positive_right\':pb.sum(0)})\n    rng=np.random.default_rng(plan[\'alignment_seed\']);null=[]\n    for _ in range(plan[\'random_alignment_replicates\']):\n        ra=rng.choice(ia,len(a),replace=False);rb=rng.choice(ib,len(b),replace=False)\n        good=valid_a[ra]&valid_b[rb]\n        null.append(float(np.sum(va[:,ra[good]]*vb[:,rb[good]],axis=0).mean()) if good.any() else np.nan)\n    summary={\'eligible_left\':len(ia),\'eligible_right\':len(ib),\'matches\':len(a),\n        \'validation_variable_matches\':int(valid.sum()),\'train_correlation_mean\':float(score[left,right].mean()),\n        \'validation_correlation_mean\':float(np.nanmean(corr)) if valid.any() else None,\n        \'decoder_cosine_mean\':float(cosine.mean()),\'validation_jaccard_mean\':float(np.nanmean(jaccard)) if np.isfinite(jaccard).any() else None,\n        \'random_alignment_validation_correlation_mean\':float(np.nanmean(null)) if np.isfinite(null).any() else None}\n    for threshold in plan[\'correlation_thresholds\']:\n        n=int(np.sum(corr>=threshold));key=str(threshold).replace(\'.\',\'p\')\n        summary.update({f\'matches_val_corr_ge_{key}\':n,f\'left_fraction_val_corr_ge_{key}\':n/len(ia),f\'right_fraction_val_corr_ge_{key}\':n/len(ib)})\n    return table,summary\n\ndef analysis_summary(root):\n    root=Path(root);records=[];points=[];stability=[]\n    plan=json.loads((root/\'protocol.json\').read_text())[\'plan\']\n    for p in root.glob(\'analysis_fits/*/encoder_*/*/sae_*/completed.json\'):\n        r=json.loads(p.read_text());identity={k:r[k] for k in [\'method\',\'encoder_seed\',\'condition\',\'sae_seed\']}\n        df=pd.read_csv(p.parent/\'bootstrap_replicates.csv\');df=df.assign(**identity);records.append(df)\n        points.append({**identity,**r[\'point\']})\n    for p in root.glob(\'stability/*/encoder_*/*/pair_*/summary.json\'):stability.append(json.loads(p.read_text()))\n    if stability:\n        sf=pd.DataFrame(stability);atomic_csv(sf,root/\'feature_stability_pairs.csv\')\n        numeric=[c for c in sf.select_dtypes(include=\'number\').columns if c not in (\'encoder_seed\',\'left_sae_seed\',\'right_sae_seed\')]\n        sg=[\'method\',\'encoder_seed\',\'condition\']\n        sm=sf.groupby(sg)[numeric].mean().reset_index();sm[\'sae_pairs\']=sf.groupby(sg).size().to_numpy()\n        atomic_csv(sm,root/\'feature_stability_per_encoder.csv\')\n        full=sm[sm.sae_pairs==len(list(itertools.combinations(plan[\'sae_seeds\'],2)))]\n        if len(full):\n            ss=full.groupby([\'method\',\'condition\'])[numeric].agg([\'mean\',\'std\',\'count\']).reset_index()\n            ss.columns=[\'_\'.join(str(x) for x in c if str(x)) for c in ss.columns]\n            atomic_csv(ss,root/\'feature_stability_summary.csv\')\n    if not records:return pd.DataFrame(),pd.DataFrame(),pd.DataFrame()\n    all_reps=pd.concat(records,ignore_index=True);pt=pd.DataFrame(points)\n    group=[\'method\',\'encoder_seed\',\'condition\']\n    sizes=pt.groupby(group).size();complete=sizes[sizes==len(plan[\'sae_seeds\'])].index\n    mask=pd.MultiIndex.from_frame(all_reps[group]).isin(complete)\n    per=all_reps[mask].groupby([*group,\'replicate\'])[BOOT_METRICS].mean().reset_index()\n    per_point=pt[ pd.MultiIndex.from_frame(pt[group]).isin(complete)].groupby(group)[BOOT_METRICS].mean().reset_index()\n    output=[]\n    def interval(label,data,point,extra):\n        for metric in BOOT_METRICS:\n            values=data[metric].to_numpy();finite=values[np.isfinite(values)]\n            if len(finite)<len(values)*.95:low=high=None\n            else:low,high=[float(v) for v in np.quantile(finite,[.025,.975])]\n            output.append({\'comparison\':label,**extra,\'metric\':metric,\'point\':float(point[metric]),\n                \'ci_low\':low,\'ci_high\':high,\'finite_replicates\':len(finite),\'total_replicates\':len(values),\n                \'scope\':\'paired patient-cluster percentile interval conditional on fixed models; exploratory validation\'})\n    for key,g in per.groupby(group):\n        p=per_point.set_index(group).loc[key];interval(\'per_encoder\',g,p,dict(zip(group,key)))\n    # Across-encoder intervals require every registered encoder, never partial groups.\n    available=per_point.groupby([\'method\',\'condition\']).encoder_seed.nunique()\n    valid_groups=available[available==len(plan[\'encoder_seeds\'])].index\n    selected=per[pd.MultiIndex.from_frame(per[[\'method\',\'condition\']]).isin(valid_groups)]\n    overall=selected.groupby([\'method\',\'condition\',\'replicate\'])[BOOT_METRICS].mean().reset_index()\n    overall_pt=per_point.groupby([\'method\',\'condition\'])[BOOT_METRICS].mean()\n    for key,g in overall.groupby([\'method\',\'condition\']):interval(\'across_fixed_encoders\',g,overall_pt.loc[key],dict(zip([\'method\',\'condition\'],key)))\n    for condition,g in overall.groupby(\'condition\'):\n        if set(g.method)!={\'ncl\',\'simclr\'}:continue\n        diff=g[g.method==\'ncl\'].set_index(\'replicate\')[BOOT_METRICS]-g[g.method==\'simclr\'].set_index(\'replicate\')[BOOT_METRICS]\n        point=overall_pt.loc[(\'ncl\',condition)]-overall_pt.loc[(\'simclr\',condition)]\n        interval(\'ncl_minus_simclr\',diff,point,{\'condition\':condition})\n    for method,g in overall.groupby(\'method\'):\n        if set(g.condition)!={\'baseline\',\'auxk\'}:continue\n        diff=g[g.condition==\'auxk\'].set_index(\'replicate\')[BOOT_METRICS]-g[g.condition==\'baseline\'].set_index(\'replicate\')[BOOT_METRICS]\n        interval(\'auxk_minus_baseline\',diff,overall_pt.loc[(method,\'auxk\')]-overall_pt.loc[(method,\'baseline\')],{\'method\':method})\n    atomic_csv(pt,root/\'bootstrap_fit_points.csv\');atomic_csv(pd.DataFrame(output),root/\'patient_bootstrap_intervals.csv\')\n    atomic_csv(per_point,root/\'per_encoder_controls.csv\')\n    return pt,per_point,pd.DataFrame(output)\n\ndef run(bundle,bundle_sha,output,plan=None,seeds=None,methods=None,max_hours=8,require_cuda=True):\n    plan=json.loads(json.dumps(DEFAULT_PLAN if plan is None else plan));seeds=plan[\'encoder_seeds\'] if seeds is None else list(seeds)\n    methods=plan[\'methods\'] if methods is None else list(methods)\n    if not seeds or set(seeds)-set(plan[\'encoder_seeds\']) or not methods or set(methods)-set(plan[\'methods\']):raise ValueError(\'Invalid selection\')\n    if require_cuda and not torch.cuda.is_available():raise RuntimeError(\'Attach a CUDA GPU\')\n    models_path=Path(plan.pop(\'models_path\',str(Path(bundle).parent/plan[\'models_filename\']))).resolve()\n    if sha(models_path)!=plan[\'models_sha256\']:raise ValueError(\'Wrong/incomplete AuxK models ZIP\')\n    root=Path(output).resolve();root.mkdir(parents=True,exist_ok=True)\n    inputs=Inputs(bundle,root/\'input_cache\',bundle_sha);device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    identity={\'bundle_sha256\':bundle_sha,\'manifest_sha256\':inputs.registration[\'manifest_sha256\'],\n        \'runtime_sha256\':globals().get(\'RUNTIME_SHA256\',\'development\'),\'plan\':plan}\n    protocol=root/\'protocol.json\'\n    if protocol.exists() and json.loads(protocol.read_text())!=identity:raise ValueError(\'Analysis protocol changed; choose new output\')\n    atomic_json(protocol,identity);atomic_json(root/\'environment.json\',environment())\n    vf=inputs.frame[inputs.frame.split==\'val\'];tf=inputs.frame[inputs.frame.split==\'train\']\n    labels=vf.dominant_label.to_numpy();patients=vf.patient_id.to_numpy();unique=sorted(set(patients))\n    if len(unique)<plan[\'top_m\']:raise ValueError(\'Too few validation patients\')\n    draws=np.random.default_rng(plan[\'bootstrap_seed\']).integers(0,len(unique),size=(plan[\'bootstrap_replicates\'],len(unique)))\n    draw_table=pd.DataFrame(draws,columns=[f\'draw_{i}\' for i in range(len(unique))]);atomic_csv(draw_table,root/\'patient_bootstrap_draw_indices.csv\')\n    atomic_json(root/\'patient_order.json\',unique)\n    train_idx=balanced_train_indices(tf,plan[\'alignment_train_slices\'],plan[\'alignment_seed\'])\n    atomic_csv(tf.iloc[train_idx][[\'sample_id\',\'patient_id\']],root/\'alignment_training_samples.csv\')\n    started=time.monotonic();status=\'completed selected controls\'\n    with zipfile.ZipFile(models_path) as archive:\n        for enc,mode in itertools.product(seeds,methods):\n            train=inputs.load(mode,enc,\'train\');val=inputs.load(mode,enc,\'val\')\n            for condition in plan[\'conditions\']:\n                activations={}\n                for sae_seed in plan[\'sae_seeds\']:\n                    if (root/\'stop_after_fit.flag\').exists() or time.monotonic()-started>=max_hours*3600:\n                        status=\'stopped between analyses; completed outputs retained\';break\n                    prefix=f\'fits/{mode}/encoder_{enc}/{condition}_r192/sae_{sae_seed}/\'\n                    metric=json.loads(archive.read(prefix+\'validation_metrics.json\'))\n                    raw=archive.read(prefix+\'completed_model.pt\')\n                    if hashlib.sha256(raw).hexdigest()!=metric[\'model_sha256\']:raise ValueError(\'Model hash mismatch\')\n                    saved=torch.load(io.BytesIO(raw),map_location=\'cpu\',weights_only=True)\n                    sp=saved[\'protocol\']\n                    if sp[\'bundle_sha256\']!=bundle_sha or sp[\'manifest_sha256\']!=inputs.registration[\'manifest_sha256\']:\n                        raise ValueError(\'Model/input identity mismatch\')\n                    if (sp[\'method\'],sp[\'encoder_seed\'],sp[\'sae_seed\'],sp[\'condition\'])!=(mode,enc,sae_seed,condition):raise ValueError(\'Model identity mismatch\')\n                    mean=saved[\'transform\'][\'mean\'].numpy()\n                    if not np.allclose(mean,train.astype(np.float64).mean(0),atol=1e-7,rtol=1e-5):raise ValueError(\'Unexpected centering\')\n                    model=SAE(192,sp[\'plan\'][\'dictionary_size\'],sp[\'plan\'][\'k\']).to(device)\n                    model.load_state_dict(saved[\'model\']);model.eval()\n                    codes,errors=encode(model,val-mean);train_codes,_=encode(model,train[train_idx]-mean)\n                    activations[sae_seed]=(train_codes,codes,saved[\'model\'][\'decoder.weight\'].numpy())\n                    folder=root/\'analysis_fits\'/mode/f\'encoder_{enc}\'/condition/f\'sae_{sae_seed}\'\n                    folder.mkdir(parents=True,exist_ok=True);done=folder/\'completed.json\'\n                    if not done.exists():\n                        reference=((val.astype(np.float64)-mean)**2).sum(1)\n                        pool=patient_pool(codes,labels,patients,errors,reference,plan[\'top_m\'],device)\n                        point=bootstrap_statistics(pool,np.arange(len(unique))[None,:],plan[\'top_m\'],1)[0]\n                        stats=bootstrap_statistics(pool,draws,plan[\'top_m\'],plan[\'bootstrap_batch\'])\n                        frame=pd.DataFrame(stats,columns=BOOT_METRICS);frame.insert(0,\'replicate\',np.arange(len(draws)))\n                        atomic_csv(frame,folder/\'bootstrap_replicates.csv\')\n                        record={\'method\':mode,\'encoder_seed\':enc,\'condition\':condition,\'sae_seed\':sae_seed,\n                            \'point\':dict(zip(BOOT_METRICS,point.tolist())),\n                            \'point_minus_archived_purity\':float(point[0]-metric[\'mean_purity\']),\n                            \'model_sha256\':metric[\'model_sha256\'],\'replicates_sha256\':sha(folder/\'bootstrap_replicates.csv\')}\n                        atomic_json(done,record);del pool\n                    else:\n                        record=json.loads(done.read_text())\n                        if record[\'model_sha256\']!=metric[\'model_sha256\'] or record[\'replicates_sha256\']!=sha(folder/\'bootstrap_replicates.csv\'):\n                            raise ValueError(\'Analysis result identity/integrity mismatch\')\n                    print(\'Bootstrap complete:\',mode,enc,condition,sae_seed,flush=True)\n                    analysis_summary(root);del model;gc.collect()\n                if status!=\'completed selected controls\':break\n                for a,b in itertools.combinations(plan[\'sae_seeds\'],2):\n                    folder=root/\'stability\'/mode/f\'encoder_{enc}\'/condition/f\'pair_{a}_{b}\'\n                    if (folder/\'summary.json\').exists():continue\n                    ta,va,da=activations[a];tb,vb,db=activations[b]\n                    table,summary=feature_match(ta,tb,va,vb,da,db,plan,device)\n                    folder.mkdir(parents=True,exist_ok=True);atomic_csv(table,folder/\'matched_features.csv\')\n                    atomic_json(folder/\'summary.json\',{\'method\':mode,\'encoder_seed\':enc,\'condition\':condition,\'left_sae_seed\':a,\'right_sae_seed\':b,**summary})\n                analysis_summary(root);del activations;gc.collect()\n            del train,val;gc.collect()\n            if status!=\'completed selected controls\':break\n    report={\'status\':status,\'encoder_seeds\':seeds,\'methods\':methods,\'test_evaluated\':False,\'models_retrained\':False,\n        \'elapsed_hours\':(time.monotonic()-started)/3600}\n    atomic_json(root/\'last_session.json\',report);analysis_summary(root);return report\n\ndef summarize(root):return analysis_summary(root)\n\ndef export_results(root):\n    root=Path(root).resolve();target=root.parent/(root.name+\'_ALL_ANALYSIS_RESULTS.zip\')\n    with zipfile.ZipFile(target,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(root.rglob(\'*\')):\n            if p.is_file() and \'input_cache\' not in p.relative_to(root).parts and p.suffix in (\'.csv\',\'.json\',\'.log\') and p.name!=\'worker_request.json\':\n                z.write(p,p.relative_to(root))\n    return str(target)\n'
engine=types.ModuleType('rank_control_runtime')
exec(compile(RUNTIME_SOURCE,'rank_control_runtime','exec'),engine.__dict__)
engine.RUNTIME_SHA256=hashlib.sha256(RUNTIME_SOURCE.encode()).hexdigest()
BUNDLE_SHA256='e41d4ae24932127a5747073d7225bdfd2d20022ca8a22027222674b7494cf9eb'


In [ ]:
start=Path.cwd().resolve()
ROOT=next((p for p in [start,*start.parents] if (p/'data/rank_control_embeddings_v1.zip').is_file()),start)
BUNDLE=ROOT/'data/rank_control_embeddings_v1.zip'
OUT=ROOT/'results/patient_stability_v1'
PLAN=json.loads(json.dumps(engine.DEFAULT_PLAN))
SELECTED_SEEDS=[10,42,52]
METHODS=['ncl','simclr']
print('saved-model analyses:',len(SELECTED_SEEDS)*len(METHODS)*len(PLAN['sae_seeds'])*len(PLAN['conditions']))
print('No model training; no raw MRI input.')


In [ ]:
MODELS_ZIP=ROOT/'data'/PLAN['models_filename']
if not MODELS_ZIP.is_file(): MODELS_ZIP=Path.home()/'Downloads'/PLAN['models_filename']
PLAN['models_path']=str(MODELS_ZIP.resolve())
print('Models archive:',MODELS_ZIP)


In [ ]:
report=engine.run(BUNDLE,BUNDLE_SHA256,OUT,PLAN,seeds=SELECTED_SEEDS,methods=METHODS,max_hours=8)
print(report)


In [ ]:
from IPython.display import display
fits,per_encoder,summary=engine.summarize(OUT)
display(per_encoder)
display(summary)
print('Mean/SD across encoders are meaningful only after all three encoder seeds finish.')


In [ ]:
archive=engine.export_results(OUT)
print('Full results and completed models:',archive)
